In [ ]:
import json
import math
import torch
from sentence_transformers import CrossEncoder
from tqdm import tqdm

# ----------------------------
# CONFIG
# ----------------------------

MODEL_NAME = "Qwen/Qwen3-Reranker-0.6B"

# ----------------------------
# FORCE GPU
# ----------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"

if device != "cuda":
    raise RuntimeError("CUDA GPU not found. This script requires GPU.")

print(f"Using device: {device}")

model = CrossEncoder(
    MODEL_NAME,
    device=device,
    max_length=512
)

# ----------------------------
# SAFE BF16 MODE (IMPORTANT FIX)
# ----------------------------

# DO NOT overwrite model.model directly
model.model.eval()
model.model.to(device)

# ----------------------------
# INSTRUCTIONS
# ----------------------------

INSTRUCTION_SET = [
    "Evaluate whether the Hindi translation preserves the meaning of the Sanskrit sentence.",
    "Check semantic equivalence between Sanskrit and Hindi translation.",
    "Judge translation adequacy: meaning preservation from Sanskrit to Hindi."
]

ALPHA_LEN_PENALTY = 0.02


# ----------------------------
# SCORING FUNCTION (SAFE AUTOCAST BF16)
# ----------------------------

def score(model, san, hin, instruction):
    text = f"{instruction}\nSanskrit: {san}\nHindi: {hin}"

    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        return model.predict([(text, "")])[0]


def rqe_single(model, san, hyp, ref):
    hyp_scores = []
    ref_scores = []

    for inst in INSTRUCTION_SET:
        s_hyp = score(model, san, hyp, inst)
        s_ref = score(model, san, ref, inst)

        hyp_scores.append(s_hyp)
        ref_scores.append(s_ref)

    s_hyp = sum(hyp_scores) / len(hyp_scores)
    s_ref = sum(ref_scores) / len(ref_scores)

    len_penalty = ALPHA_LEN_PENALTY * abs(len(hyp) - len(ref))
    s_hyp = s_hyp - len_penalty

    diff = s_hyp - s_ref

    rqe = torch.sigmoid(
        torch.tensor(diff, device=device, dtype=torch.bfloat16)
    ).item()

    return {
        "s_hyp": round(float(s_hyp), 6),
        "s_ref": round(float(s_ref), 6),
        "diff": round(float(diff), 6),
        "rqe_0_1": round(rqe, 6),
        "rqe_0_100": round(rqe * 100, 2)
    }


# ----------------------------
# LOAD DATA
# ----------------------------

def load_jsonl(file_path):
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))
    return data


# ----------------------------
# EVALUATION WITH PROGRESS BAR
# ----------------------------

def evaluate(data):
    results = []

    for row in tqdm(data, desc="Scoring rows", unit="row"):
        san = row["san"]
        ref = row["hin"]
        hyp = row["gen"]

        metrics = rqe_single(model, san, hyp, ref)

        results.append({
            **row,
            **metrics
        })

    return results


# ----------------------------
# SUMMARY
# ----------------------------

def dataset_summary(results):
    scores = [r["rqe_0_100"] for r in results]

    return {
        "count": len(scores),
        "mean_rqe": round(sum(scores) / len(scores), 2),
        "min_rqe": round(min(scores), 2),
        "max_rqe": round(max(scores), 2)
    }


# ----------------------------
# SAVE UTILITIES
# ----------------------------

def save_jsonl(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for row in data:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def save_summary(data, path):
    with open(path, "w", encoding="utf-8") as f:
        f.write(json.dumps(data, ensure_ascii=False, indent=2))


# ----------------------------
# MAIN
# ----------------------------

if __name__ == "__main__":

    input_file = "/content/drive/MyDrive/sanskrit/eval-metric/nllb200_1p3b_outputs.jsonl"

    row_output_path = "/content/drive/MyDrive/sanskrit/eval-metric/nllb200_1p3b_score_rows.jsonl"
    summary_output_path = "/content/drive/MyDrive/sanskrit/eval-metric/nllb200_1p3b_score_summary.jsonl"

    data = load_jsonl(input_file)

    print(f"Processing dataset size={len(data)}")

    results = evaluate(data)

    summary = dataset_summary(results)

    save_jsonl(results, row_output_path)
    save_summary(summary, summary_output_path)

    print("Done.")

In [ ]:
import subprocess
import time

while True:
    output = subprocess.check_output(
        [
            "nvidia-smi",
            "--query-gpu=memory.used,memory.total,utilization.gpu",
            "--format=csv,noheader,nounits"
        ]
    ).decode("utf-8")

    used, total, util = output.strip().split(", ")

    used_gb = int(used) / 1024
    total_gb = int(total) / 1024

    print(f"GPU: {used_gb:.2f} GB / {total_gb:.2f} GB | Util: {util}%")

    time.sleep(1)

GPU: 2.50 GB / 15.00 GB | Util: 0%
GPU: 2.50 GB / 15.00 GB | Util: 0%
GPU: 2.50 GB / 15.00 GB | Util: 0%
GPU: 2.50 GB / 15.00 GB | Util: 0%
GPU: 2.50 GB / 15.00 GB | Util: 0%
GPU: 2.50 GB / 15.00 GB | Util: 0%
GPU: 2.50 GB / 15.00 GB | Util: 0%
GPU: 2.50 GB / 15.00 GB | Util: 0%
GPU: 2.50 GB / 15.00 GB | Util: 0%
GPU: 2.50 GB / 15.00 GB | Util: 0%
GPU: 2.50 GB / 15.00 GB | Util: 0%
GPU: 2.50 GB / 15.00 GB | Util: 0%
GPU: 2.50 GB / 15.00 GB | Util: 0%
GPU: 2.50 GB / 15.00 GB | Util: 0%
GPU: 2.50 GB / 15.00 GB | Util: 0%
GPU: 2.50 GB / 15.00 GB | Util: 0%


KeyboardInterrupt: 

In [ ]:
import json
import math
import torch
from sentence_transformers import CrossEncoder
from tqdm import tqdm

# >>> ADDED FOR GPU MONITORING
import threading
import subprocess
import time

# ----------------------------
# CONFIG
# ----------------------------

MODEL_NAME = "Qwen/Qwen3-Reranker-0.6B"

# ----------------------------
# GPU MONITOR (ADDED)
# ----------------------------

def gpu_monitor():
    while True:
        try:
            output = subprocess.getoutput(
                "nvidia-smi --query-gpu=memory.used,memory.total,utilization.gpu --format=csv,noheader,nounits"
            )

            used, total, util = output.strip().split(", ")

            used_gb = int(used) / 1024
            total_gb = int(total) / 1024

            print(f"[GPU] {used_gb:.2f}/{total_gb:.2f} GB | {util}% util")

        except Exception as e:
            print("[GPU MONITOR ERROR]", e)

        time.sleep(5)

# ----------------------------
# FORCE GPU
# ----------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"

if device != "cuda":
    raise RuntimeError("CUDA GPU not found. This script requires GPU.")

print(f"Using device: {device}")

model = CrossEncoder(
    MODEL_NAME,
    device=device,
    max_length=512
)

# ----------------------------
# SAFE BF16 MODE (IMPORTANT FIX)
# ----------------------------

# DO NOT overwrite model.model directly
model.model.eval()
model.model.to(device)

# ----------------------------
# INSTRUCTIONS
# ----------------------------

INSTRUCTION_SET = [
    "Evaluate whether the Hindi translation preserves the meaning of the Sanskrit sentence.",
    "Check semantic equivalence between Sanskrit and Hindi translation.",
    "Judge translation adequacy: meaning preservation from Sanskrit to Hindi."
]

ALPHA_LEN_PENALTY = 0.02


# ----------------------------
# SCORING FUNCTION (SAFE AUTOCAST BF16)
# ----------------------------

def score(model, san, hin, instruction):
    text = f"{instruction}\nSanskrit: {san}\nHindi: {hin}"

    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        return model.predict([(text, "")])[0]


def rqe_single(model, san, hyp, ref):
    hyp_scores = []
    ref_scores = []

    for inst in INSTRUCTION_SET:
        s_hyp = score(model, san, hyp, inst)
        s_ref = score(model, san, ref, inst)

        hyp_scores.append(s_hyp)
        ref_scores.append(s_ref)

    s_hyp = sum(hyp_scores) / len(hyp_scores)
    s_ref = sum(ref_scores) / len(ref_scores)

    len_penalty = ALPHA_LEN_PENALTY * abs(len(hyp) - len(ref))
    s_hyp = s_hyp - len_penalty

    diff = s_hyp - s_ref

    rqe = torch.sigmoid(
        torch.tensor(diff, device=device, dtype=torch.bfloat16)
    ).item()

    return {
        "s_hyp": round(float(s_hyp), 6),
        "s_ref": round(float(s_ref), 6),
        "diff": round(float(diff), 6),
        "rqe_0_1": round(rqe, 6),
        "rqe_0_100": round(rqe * 100, 2)
    }


# ----------------------------
# LOAD DATA
# ----------------------------

def load_jsonl(file_path):
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))
    return data


# ----------------------------
# EVALUATION WITH PROGRESS BAR
# ----------------------------

def evaluate(data):
    results = []

    for row in tqdm(data, desc="Scoring rows", unit="row"):
        san = row["san"]
        ref = row["hin"]
        hyp = row["gen"]

        metrics = rqe_single(model, san, hyp, ref)

        results.append({
            **row,
            **metrics
        })

    return results


# ----------------------------
# SUMMARY
# ----------------------------

def dataset_summary(results):
    scores = [r["rqe_0_100"] for r in results]

    return {
        "count": len(scores),
        "mean_rqe": round(sum(scores) / len(scores), 2),
        "min_rqe": round(min(scores), 2),
        "max_rqe": round(max(scores), 2)
    }


# ----------------------------
# SAVE UTILITIES
# ----------------------------

def save_jsonl(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for row in data:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def save_summary(data, path):
    with open(path, "w", encoding="utf-8") as f:
        f.write(json.dumps(data, ensure_ascii=False, indent=2))


# ----------------------------
# MAIN
# ----------------------------

if __name__ == "__main__":

    # >>> START GPU MONITOR THREAD (ADDED HERE)
    monitor_thread = threading.Thread(target=gpu_monitor, daemon=True)
    monitor_thread.start()

    input_file = "/content/drive/MyDrive/sanskrit/eval-metric/nllb200_1p3b_outputs.jsonl"

    row_output_path = "/content/drive/MyDrive/sanskrit/eval-metric/nllb200_1p3b_score_rows.jsonl"
    summary_output_path = "/content/drive/MyDrive/sanskrit/eval-metric/nllb200_1p3b_score_summary.jsonl"

    data = load_jsonl(input_file)

    print(f"Processing dataset size={len(data)}")

    results = evaluate(data)

    summary = dataset_summary(results)

    save_jsonl(results, row_output_path)
    save_summary(summary, summary_output_path)

    print("Done.")

Using device: cuda


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

[GPU] 3.55/15.00 GB | 18% util
Processing dataset size=5282


Scoring rows:   0%|          | 3/5282 [00:03<1:54:01,  1.30s/row]

[GPU] 3.61/15.00 GB | 97% util


...

[GPU] 3.61/15.00 GB | 94% util


Scoring rows:   3%|▎         | 170/5282 [03:00<1:39:41,  1.17s/row]

[GPU] 3.61/15.00 GB | 94% util


Scoring rows:   3%|▎         | 173/5282 [03:03<1:30:23,  1.06s/row]


KeyboardInterrupt: 

In [ ]:
import json
import math
import torch
from sentence_transformers import CrossEncoder
from tqdm import tqdm

import threading
import subprocess
import time

# ----------------------------
# CONFIG
# ----------------------------

MODEL_NAME = "Qwen/Qwen3-Reranker-0.6B"

# >>> ADDED: batch size (safe default for CrossEncoder GPU inference)
BATCH_SIZE = 1
# ----------------------------
# GPU MONITOR (UPDATED PRINT STYLE ONLY)
# ----------------------------

def gpu_monitor():
    while True:
        try:
            output = subprocess.getoutput(
                "nvidia-smi --query-gpu=memory.used,memory.total,utilization.gpu --format=csv,noheader,nounits"
            )

            used, total, util = output.strip().split(", ")

            used_gb = int(used) / 1024
            total_gb = int(total) / 1024

            # >>> CHANGED: use tqdm-safe print (no line collision)
            tqdm.write(f"[GPU] {used_gb:.2f}/{total_gb:.2f} GB | {util}% util")

        except Exception as e:
            tqdm.write(f"[GPU MONITOR ERROR] {e}")

        time.sleep(5)

# ----------------------------
# DEVICE
# ----------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"

if device != "cuda":
    raise RuntimeError("CUDA GPU not found. This script requires GPU.")

print(f"Using device: {device}")

model = CrossEncoder(
    MODEL_NAME,
    device=device,
    max_length=512
)

model.model.eval()
model.model.to(device)

# ----------------------------
# INSTRUCTIONS
# ----------------------------

INSTRUCTION_SET = [
    "Evaluate whether the Hindi translation preserves the meaning of the Sanskrit sentence.",
    "Check semantic equivalence between Sanskrit and Hindi translation.",
    "Judge translation adequacy: meaning preservation from Sanskrit to Hindi."
]

ALPHA_LEN_PENALTY = 0.02


# ----------------------------
# SCORING FUNCTION (UNCHANGED)
# ----------------------------

def score(model, san, hin, instruction):
    text = f"{instruction}\nSanskrit: {san}\nHindi: {hin}"

    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        return model.predict([(text, "")])[0]


def rqe_single(model, san, hyp, ref):
    hyp_scores = []
    ref_scores = []

    for inst in INSTRUCTION_SET:
        s_hyp = score(model, san, hyp, inst)
        s_ref = score(model, san, ref, inst)

        hyp_scores.append(s_hyp)
        ref_scores.append(s_ref)

    s_hyp = sum(hyp_scores) / len(hyp_scores)
    s_ref = sum(ref_scores) / len(ref_scores)

    len_penalty = ALPHA_LEN_PENALTY * abs(len(hyp) - len(ref))
    s_hyp = s_hyp - len_penalty

    diff = s_hyp - s_ref

    rqe = torch.sigmoid(
        torch.tensor(diff, device=device, dtype=torch.bfloat16)
    ).item()

    return {
        "s_hyp": round(float(s_hyp), 6),
        "s_ref": round(float(s_ref), 6),
        "diff": round(float(diff), 6),
        "rqe_0_1": round(rqe, 6),
        "rqe_0_100": round(rqe * 100, 2)
    }


# ----------------------------
# LOAD DATA
# ----------------------------

def load_jsonl(file_path):
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))
    return data


# ----------------------------
# EVALUATION (UPDATED: STREAMING + BATCH SAFE)
# ----------------------------

def evaluate(data, output_path):

    results = []

    # >>> ADDED: open file once (prevents CPU overhead per row)
    with open(output_path, "w", encoding="utf-8") as f:

        for i in tqdm(range(0, len(data), BATCH_SIZE), desc="Scoring rows", unit="batch"):

            batch = data[i:i + BATCH_SIZE]

            for row in batch:
                san = row["san"]
                ref = row["hin"]
                hyp = row["gen"]

                metrics = rqe_single(model, san, hyp, ref)

                out = {
                    **row,
                    **metrics
                }

                results.append(out)

                # >>> ADDED: immediate disk append (NO final CPU dump)
                f.write(json.dumps(out, ensure_ascii=False) + "\n")
                f.flush()

    return results


# ----------------------------
# SUMMARY
# ----------------------------

def dataset_summary(results):
    scores = [r["rqe_0_100"] for r in results]

    return {
        "count": len(scores),
        "mean_rqe": round(sum(scores) / len(scores), 2),
        "min_rqe": round(min(scores), 2),
        "max_rqe": round(max(scores), 2)
    }


def save_summary(data, path):
    with open(path, "w", encoding="utf-8") as f:
        f.write(json.dumps(data, ensure_ascii=False, indent=2))


# ----------------------------
# MAIN
# ----------------------------

if __name__ == "__main__":

    monitor_thread = threading.Thread(target=gpu_monitor, daemon=True)
    monitor_thread.start()

    input_file = "/content/drive/MyDrive/sanskrit/eval-metric/nllb200_1p3b_outputs.jsonl"

    row_output_path = "/content/drive/MyDrive/sanskrit/eval-metric/nllb200_1p3b_score_rows.jsonl"
    summary_output_path = "/content/drive/MyDrive/sanskrit/eval-metric/nllb200_1p3b_score_summary.jsonl"

    data = load_jsonl(input_file)

    print(f"Processing dataset size={len(data)}")

    # >>> CHANGED: streaming writer inside evaluate
    results = evaluate(data, row_output_path)

    summary = dataset_summary(results)

    save_summary(summary, summary_output_path)

    print("Done.")

Using device: cuda
[GPU] 3.67/15.00 GB | 0% util


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

[GPU] 3.67/15.00 GB | 0% util
[GPU] 3.67/15.00 GB | 0% util
[GPU] 3.67/15.00 GB | 0% util
[GPU] 3.67/15.00 GB | 26% util
Processing dataset size=5282


Scoring rows:   0%|          | 1/5282 [00:00<1:18:42,  1.12batch/s]

[GPU] 3.67/15.00 GB | 96% util


...

[GPU] 3.67/15.00 GB | 95% util


Scoring rows: 100%|█████████▉| 5280/5282 [1:15:04<00:01,  1.25batch/s]

[GPU] 3.67/15.00 GB | 71% util
[GPU] 3.67/15.00 GB | 95% util


Scoring rows: 100%|██████████| 5282/5282 [1:15:06<00:00,  1.17batch/s]

Done.


In [ ]:
import json
import math
import torch
from sentence_transformers import CrossEncoder
from tqdm import tqdm

import threading
import subprocess
import time

# ----------------------------
# CONFIG
# ----------------------------

MODEL_NAME = "Qwen/Qwen3-Reranker-0.6B"

# >>> ADDED: batch size (safe default for CrossEncoder GPU inference)
BATCH_SIZE = 1
# ----------------------------
# GPU MONITOR (UPDATED PRINT STYLE ONLY)
# ----------------------------

def gpu_monitor():
    while True:
        try:
            output = subprocess.getoutput(
                "nvidia-smi --query-gpu=memory.used,memory.total,utilization.gpu --format=csv,noheader,nounits"
            )

            used, total, util = output.strip().split(", ")

            used_gb = int(used) / 1024
            total_gb = int(total) / 1024

            # >>> CHANGED: use tqdm-safe print (no line collision)
            tqdm.write(f"[GPU] {used_gb:.2f}/{total_gb:.2f} GB | {util}% util")

        except Exception as e:
            tqdm.write(f"[GPU MONITOR ERROR] {e}")

        time.sleep(5)

# ----------------------------
# DEVICE
# ----------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"

if device != "cuda":
    raise RuntimeError("CUDA GPU not found. This script requires GPU.")

print(f"Using device: {device}")

model = CrossEncoder(
    MODEL_NAME,
    device=device,
    max_length=512
)

model.model.eval()
model.model.to(device)

# ----------------------------
# INSTRUCTIONS
# ----------------------------

INSTRUCTION_SET = [
    "Evaluate whether the Hindi translation preserves the meaning of the Sanskrit sentence.",
    "Check semantic equivalence between Sanskrit and Hindi translation.",
    "Judge translation adequacy: meaning preservation from Sanskrit to Hindi."
]

ALPHA_LEN_PENALTY = 0.02


# ----------------------------
# SCORING FUNCTION (UNCHANGED)
# ----------------------------

def score(model, san, hin, instruction):
    text = f"{instruction}\nSanskrit: {san}\nHindi: {hin}"

    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        return model.predict([(text, "")])[0]


def rqe_single(model, san, hyp, ref):
    hyp_scores = []
    ref_scores = []

    for inst in INSTRUCTION_SET:
        s_hyp = score(model, san, hyp, inst)
        s_ref = score(model, san, ref, inst)

        hyp_scores.append(s_hyp)
        ref_scores.append(s_ref)

    s_hyp = sum(hyp_scores) / len(hyp_scores)
    s_ref = sum(ref_scores) / len(ref_scores)

    len_penalty = ALPHA_LEN_PENALTY * abs(len(hyp) - len(ref))
    s_hyp = s_hyp - len_penalty

    diff = s_hyp - s_ref

    rqe = torch.sigmoid(
        torch.tensor(diff, device=device, dtype=torch.bfloat16)
    ).item()

    return {
        "s_hyp": round(float(s_hyp), 6),
        "s_ref": round(float(s_ref), 6),
        "diff": round(float(diff), 6),
        "rqe_0_1": round(rqe, 6),
        "rqe_0_100": round(rqe * 100, 2)
    }


# ----------------------------
# LOAD DATA
# ----------------------------

def load_jsonl(file_path):
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))
    return data


# ----------------------------
# EVALUATION (UPDATED: STREAMING + BATCH SAFE)
# ----------------------------

def evaluate(data, output_path):

    results = []

    # >>> ADDED: open file once (prevents CPU overhead per row)
    with open(output_path, "w", encoding="utf-8") as f:

        for i in tqdm(range(0, len(data), BATCH_SIZE), desc="Scoring rows", unit="batch"):

            batch = data[i:i + BATCH_SIZE]

            for row in batch:
                san = row["san"]
                ref = row["hin"]
                hyp = row["gen"]

                metrics = rqe_single(model, san, hyp, ref)

                out = {
                    **row,
                    **metrics
                }

                results.append(out)

                # >>> ADDED: immediate disk append (NO final CPU dump)
                f.write(json.dumps(out, ensure_ascii=False) + "\n")
                f.flush()

    return results


# ----------------------------
# SUMMARY
# ----------------------------

def dataset_summary(results):
    scores = [r["rqe_0_100"] for r in results]

    return {
        "count": len(scores),
        "mean_rqe": round(sum(scores) / len(scores), 2),
        "min_rqe": round(min(scores), 2),
        "max_rqe": round(max(scores), 2)
    }


def save_summary(data, path):
    with open(path, "w", encoding="utf-8") as f:
        f.write(json.dumps(data, ensure_ascii=False, indent=2))


# ----------------------------
# MAIN
# ----------------------------

if __name__ == "__main__":

    monitor_thread = threading.Thread(target=gpu_monitor, daemon=True)
    monitor_thread.start()

    input_file = "/content/drive/MyDrive/sanskrit/eval-metric/nllb200_3p3b_outputs.jsonl"

    row_output_path = "/content/drive/MyDrive/sanskrit/eval-metric/nllb200_3p3b_score_rows.jsonl"
    summary_output_path = "/content/drive/MyDrive/sanskrit/eval-metric/nllb200_3p3b_score_summary.jsonl"

    data = load_jsonl(input_file)

    print(f"Processing dataset size={len(data)}")

    # >>> CHANGED: streaming writer inside evaluate
    results = evaluate(data, row_output_path)

    summary = dataset_summary(results)

    save_summary(summary, summary_output_path)

    print("Done.")

Using device: cuda
[GPU] 3.67/15.00 GB | 0% util


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

[GPU] 3.67/15.00 GB | 0% util
[GPU] 3.67/15.00 GB | 0% util
[GPU] 3.67/15.00 GB | 0% util
[GPU] 3.67/15.00 GB | 0% util
[GPU] 3.67/15.00 GB | 26% util
Processing dataset size=5282


Scoring rows:   0%|          | 0/5282 [00:00<?, ?batch/s]

[GPU] 3.67/15.00 GB | 97% util


...

[GPU] 3.67/15.00 GB | 91% util


Scoring rows: 100%|█████████▉| 5279/5282 [1:15:14<00:02,  1.28batch/s]

[GPU] 3.67/15.00 GB | 93% util
[GPU] 3.67/15.00 GB | 93% util


Scoring rows: 100%|██████████| 5282/5282 [1:15:15<00:00,  1.17batch/s]

Done.
